# Monnaie et prix sur le long terme · *Money and prices over the long run*

Notebook compagnon du chapitre **26. La théorie quantitative de la monnaie (MV = PQ) : relier monnaie, prix et activité** — [lire l'article](https://nmlab.io/ressources/theorie-quantitative-monnaie-mv-pq).
Companion notebook to chapter **26. The Quantity Theory of Money (MV = PQ): Linking Money, Prices and Activity** — [read the article](https://nmlab.io/en/ressources/quantity-theory-of-money).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# (les séries sont chargées dans build_figure)


import numpy as np
import pandas as pd
from matplotlib.figure import Figure
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

C, W = nm.COLORS, nm.WIDTH_PX

EA_M3_KEY = "BSI/M.U2.Y.V.M30.X.1.U2.2300.Z01.E"   # encours M3 zone euro (BCE)


def load(series_id: str, start: str | None = None, end: str | None = None) -> pd.Series:
    """Charge une série en direct : FRED, ou le portail de la BCE pour « EA_M3 »."""
    if series_id == "EA_M3":
        url = (f"https://data-api.ecb.europa.eu/service/data/{EA_M3_KEY}"
               "?format=csvdata&detail=dataonly")
        raw = pd.read_csv(url)
        s = pd.Series(raw["OBS_VALUE"].values,
                      index=pd.PeriodIndex(raw["TIME_PERIOD"], freq="M").to_timestamp())
        s = s.sort_index() / 1000.0                 # millions -> milliards d'euros
    else:
        s = nm.load_fred(series_id)
    return s.loc[start:end]


def T(d: dict, lang: str):
    """Sélectionne le jeu de libellés de la langue demandée."""
    return d[lang]


def build_figure(lang: str = "fr") -> Figure:
    """Construit la figure NMLab du chapitre (libellés selon ``lang``)."""
    m2=load("M2SL","1959-01"); cpi=load("CPIAUCSL","1959-01")
    m2i=100*m2/m2.iloc[0]; cpii=100*cpi/cpi.iloc[0]
    fig=nm.figure(1010); ax=nm.axes(fig)
    ax.plot(m2i.index,m2i.values,color=C["blue"],lw=3,label=("Monnaie (M2)" if lang=="fr" else "Money (M2)"))
    ax.plot(cpii.index,cpii.values,color=C["amber"],lw=3,label=("Prix (IPC)" if lang=="fr" else "Prices (CPI)"))
    ax.set_yscale("log")
    d=dict(fr=("Sur le long terme, monnaie et prix montent ensemble","États-Unis, indice base 100 en 1959, échelle logarithmique.",
               "× 80","× 11","Le lien est réel — mais pas de un à un : l'écart part dans la production (Q) et la baisse de la vitesse (V).\nSource : FRED (M2SL, CPIAUCSL)."),
           en=("Over the long run, money and prices rise together","United States, index 100 in 1959, logarithmic scale.",
               "× 80","× 11","The link is real — but not one-for-one: the gap goes into output (Q) and the fall in velocity (V).\nSource: FRED (M2SL, CPIAUCSL)."))
    t=T(d,lang); nm.header(fig,t[0],t[1])
    ax.text(m2i.index[-1],m2i.iloc[-1]*1.15,t[2],color=C["blue"],fontsize=20,fontweight="bold",ha="right",va="bottom")
    ax.text(cpii.index[-1],cpii.iloc[-1]*0.72,t[3],color=C["amber"],fontsize=20,fontweight="bold",ha="right",va="top")
    leg=ax.legend(loc="upper left",fontsize=18,frameon=True,facecolor=C["bg"],edgecolor=C["edge"])
    for txt in leg.get_texts(): txt.set_color(C["text"])
    nm.footer(fig,t[4]);
    return fig


build_figure(LANG)